## **설정**

In [1]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
## Import libaries
import os

import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

In [3]:
## Setting
base_path = '/content/drive/MyDrive/CS/'

## **데이터 불러오기**

In [4]:
file_path = os.path.join(base_path, 'hidden_states', 'CNN_Firm_simple')
years = [2018, 2019, 2020, 2021]

firm_sizes = [5, 10, 20, 50]
data_types = ['train', 'valid', 'test']

hidden_states_data = {}

for firm_size in firm_sizes:
    hidden_states_data[firm_size] = {}
    for data_type in data_types:
        hidden_states_data[firm_size][data_type] = {}
        for year in years:
            folder_name = f'Test_{year}'
            file_name = f'hidden_states_{data_type}_{firm_size}_Firm.csv'
            full_file_path = os.path.join(file_path, folder_name, file_name)
            hidden_states_data[firm_size][data_type][year] = pd.read_csv(full_file_path)

In [5]:
print(hidden_states_data[5]['train'][2018].head())

  ticker  gvkey  permno   sic  exchcd  shrcd  ffi49  year       ret  \
0    AAL   1045   21020  4512     3.0   11.0     41  1997  0.127273   
1   AAPL   1690   14593  3663     3.0   11.0     37  1997 -0.022556   
2   AAPL   1690   14593  3663     3.0   11.0     37  1997  0.123077   
3   AAPL   1690   14593  3663     3.0   11.0     37  1997 -0.068493   
4   AAPL   1690   14593  3663     3.0   11.0     37  1997 -0.022059   

         date    hs_5_0    hs_5_1    hs_5_2    hs_5_3    hs_5_4  
0  1997-04-30 -0.010835 -0.058082 -0.003436  0.272863  0.000140  
1  1997-02-28 -0.019922 -0.020855 -0.029816  0.190561  0.230301  
2  1997-03-31 -0.023130 -0.031718 -0.033140  0.072414  0.396948  
3  1997-04-30 -0.021418 -0.033557 -0.030846  0.088695  0.363819  
4  1997-05-31 -0.023939 -0.024360 -0.033392  0.099934  0.382672  


In [6]:
def set_seed(val):
    torch.manual_seed(val)
    torch.cuda.manual_seed(val)
    np.random.seed(val)
    random.seed(val)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 저장 경로 설정
SAVE_BASE_DIR = os.path.join(base_path, 'hidden_states', 'CNNLSTM_Firm_simple')

# 메타 데이터 컬럼 리스트
meta_cols = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'year', 'ret', 'date']

# 입력 크기별 출력 타겟 매핑
# Input Size(CNN Output) -> [Output Sizes(LSTM Output)]
size_mapping = {
    5: [5],
    10: [5, 10],
    20: [5, 10, 20],
    50: [5, 10, 20, 50]
}

In [7]:
# --- 2. Multi-Head Bi-LSTM Encoder 모델 정의 ---
class MultiHeadBiLSTMEncoder(nn.Module):
    def __init__(self, input_dim, output_dims):
        super(MultiHeadBiLSTMEncoder, self).__init__()
        self.output_dims = output_dims

        # Shared Bi-LSTM
        # 1:1 매핑이므로 Sequence Length = 1로 들어옴
        # 이 경우 LSTM은 복잡한 Dense Layer처럼 동작함 (Context 정보 X)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=input_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Projections
        self.heads = nn.ModuleList()
        for out_dim in output_dims:
            # Bi-LSTM output (input_dim * 2) -> Target dim
            self.heads.append(nn.Linear(input_dim * 2, out_dim))

    def forward(self, x):
        # x shape: (Batch, 1, Input_Dim)

        # LSTM 통과
        # output shape: (Batch, 1, Hidden*2)
        lstm_out, _ = self.lstm(x)

        # 차원 축소: (Batch, Hidden*2)
        last_step_feature = lstm_out[:, -1, :]

        results = {}
        for i, head in enumerate(self.heads):
            # Projection
            out = head(last_step_feature)
            results[f'hs_{self.output_dims[i]}'] = out

        return results

In [8]:
# firm_sizes 루프 (5, 10, 20, 50)
for firm_size in firm_sizes:
    target_dims = size_mapping[firm_size]
    print(f"\n[Processing Firm Size: {firm_size} -> Targets: {target_dims}]")

    # 모델 초기화 (입력 차원 = firm_size)
    model = MultiHeadBiLSTMEncoder(input_dim=firm_size, output_dims=target_dims)
    model.eval()

    # Data Type 루프 (train, valid, test)
    for data_type in data_types:
        for year in years:
            # 1. 데이터 가져오기
            try:
                df = hidden_states_data[firm_size][data_type][year]
            except KeyError:
                continue

            if df.empty:
                continue

            # 2. Feature와 Meta 분리
            current_features = [c for c in df.columns if c not in meta_cols]

            # 혹시 모를 정렬
            if 'date' in df.columns:
                df = df.sort_values('date').reset_index(drop=True)

            # Feature 추출
            feature_data = df[current_features].values

            # 3. 텐서 변환 (Windowing 없음)
            # LSTM 입력 규격 (Batch, Sequence, Input_Dim)을 맞추기 위해
            # Sequence Length를 1로 설정 -> (N, 1, Feature_Dim)
            X_tensor = torch.FloatTensor(feature_data).unsqueeze(1)

            # 메타 데이터는 그대로 사용 (행 개수 변화 없음)
            meta_data = df[meta_cols].copy()

            # 4. 모델 실행
            with torch.no_grad():
                outputs = model(X_tensor)

            # 5. 저장
            save_folder = os.path.join(SAVE_BASE_DIR, f'Test_{year}')
            os.makedirs(save_folder, exist_ok=True)

            for dim in target_dims:
                # 추출된 Hidden State
                feats = outputs[f'hs_{dim}'].numpy()
                feat_cols = [f'hs_{dim}_{k}' for k in range(dim)]

                # DataFrame 생성
                res_df = pd.DataFrame(feats, columns=feat_cols)

                # 메타 데이터 붙이기 (1:1 매핑이므로 바로 concat)
                res_df = pd.concat([meta_data, res_df], axis=1)

                # 파일명 설정
                file_name = f'hidden_states_{data_type}_{firm_size}_{dim}_Firm.csv'
                save_path = os.path.join(save_folder, file_name)

                res_df.to_csv(save_path, index=False)

    print(f" -> Firm Size {firm_size} 완료")

print("\nAll tasks completed.")


[Processing Firm Size: 5 -> Targets: [5]]
 -> Firm Size 5 완료

[Processing Firm Size: 10 -> Targets: [5, 10]]
 -> Firm Size 10 완료

[Processing Firm Size: 20 -> Targets: [5, 10, 20]]
 -> Firm Size 20 완료

[Processing Firm Size: 50 -> Targets: [5, 10, 20, 50]]
 -> Firm Size 50 완료

All tasks completed.
